# 09 检索增强生成 (RAG) · 代码示例

**章节**: 09_rag.md
**学习目标**: 跑通一个 TF-IDF 简化版 RAG 检索器
**A3 关联**: A3 系统的「知识库溯源」功能核心

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 模拟 A3 课程知识库 chunk
documents = [
    "Transformer 使用 Self-Attention 机制实现并行计算,解决 RNN 长程遗忘。",
    "过拟合指模型在训练集表现好但在测试集表现差,可通过正则化缓解。",
    "多智能体系统由 Planner、Tutor、Evaluator 等多个 Agent 协作完成任务。",
    "RAG 通过检索外部知识库来缓解大模型幻觉,并支持答案溯源。",
    "BKT 模型用贝叶斯更新追踪学生在每个知识点上的掌握度。"
]

def rag_retrieve(query, docs, top_k=2):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform(docs + [query])
    doc_vecs = tfidf[:-1]
    query_vec = tfidf[-1]
    sims = cosine_similarity(query_vec, doc_vecs).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(docs[i], sims[i]) for i in top_idx]

### 解读

- 把所有文档和查询一起向量化,确保在同一个词表空间
- 计算查询向量与每个文档的余弦相似度
- 取 top_k 个最相关的文档作为召回结果

In [ ]:
queries = [
    "什么是 Self-Attention?",
    "如何缓解过拟合?",
    "A3 怎么用多个 AI 协作?",
    "大模型为什么会胡说八道?",
    "BKT 是怎么工作的?"
]

for q in queries:
    print(f"\n【提问】: {q}")
    results = rag_retrieve(q, documents)
    for doc, score in results:
        print(f"  [相似度 {score:.3f}] {doc}")

### 拓展练习

1. 改成基于 sentence-transformers 的版本,对比效果
2. 加入 chunk overlap 切分,模拟真实 Markdown 章节切块
3. 把召回结果塞进 Prompt,实现完整 RAG 闭环(不调大模型,只打印 prompt)